In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score

DATA = "../data/raw/sample_23.csv"
ID_COL = "ID"

CLUSTERINGS = [
    # 'outputs/shapelet_experiments/shapelet_cluster_labels.csv',
    # 'outputs/kshape/kshape_7.csv',
    # 'outputs/kshape/kshape_7_znorm.csv',
    # 'outputs/kmedoid/kmedoid_cluster_labels_sample_23.csv',
    # 'outputs/kmedoid/kmedoid_cluster_labels_sample_23_znorm.csv',
    # 'outputs/feature/clusters.csv',
    # 'outputs/feature/clusters_named.csv',
    # 'outputs/custom/custom_clustering_best_mae.csv'
    '../data/outputs/agp_forecast_case5_clusters_cluster_specific_combined.csv',
    # 'outputs/feature/case5_clusters.csv',
]

In [2]:
def load_cluster_labels(path):
    labels_df = pd.read_csv(path)
    labels_df.columns = labels_df.columns.str.strip()

    id_candidates = [c for c in labels_df.columns if c.lower() == ID_COL.lower()]
    cluster_candidates = [c for c in labels_df.columns if "cluster" in c.lower()]

    id_col = id_candidates[0] if id_candidates else labels_df.columns[0]
    if not cluster_candidates and len(labels_df.columns) < 2:
        raise ValueError(f"Could not find a cluster label column in {path}")
    cluster_col = cluster_candidates[0] if cluster_candidates else labels_df.columns[1]

    labels_df = labels_df[[id_col, cluster_col]].dropna().copy()
    labels_df[id_col] = labels_df[id_col].astype(str)
    labels_df = labels_df.rename(columns={id_col: ID_COL, cluster_col: "cluster_label"})
    labels_df = labels_df.drop_duplicates(subset=[ID_COL], keep="last")
    return labels_df

In [3]:
import math

import matplotlib.pyplot as plt


def plot_clusters_with_centroids(clustering, alpha=0.15, lw=0.7, centroid_lw=2.5):
    """Plot each cluster in its own subplot within a single figure."""
    labels_df = label_vectors.get(clustering)
    if labels_df is None:
        if "load_cluster_labels" not in globals():
            raise ValueError(f"No labels found for clustering: {clustering}")
        labels_df = load_cluster_labels(clustering)

    merged = X_df[[ID_COL]].merge(labels_df[[ID_COL, "cluster_label"]], on=ID_COL, how="inner")
    if merged.empty:
        raise ValueError("No overlapping IDs between data and clustering labels.")

    X = X_df.set_index(ID_COL).loc[merged[ID_COL], date_cols].to_numpy(dtype=float)
    cluster_labels = merged["cluster_label"].to_numpy()
    clusters = sorted(pd.unique(cluster_labels))
    x = pd.to_datetime(date_cols)

    n_clusters = len(clusters)
    ncols = 3 if n_clusters > 1 else 1
    nrows = math.ceil(n_clusters / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), sharex=True, sharey=False)
    axes = np.atleast_1d(axes).ravel()
    colors = plt.cm.tab20(np.linspace(0, 1, max(n_clusters, 1)))

    for ax, cluster, color in zip(axes, clusters, colors):
        mask = cluster_labels == cluster
        cluster_X = X[mask]

        for ts in cluster_X:
            ax.plot(x, ts, color=color, alpha=alpha, linewidth=lw)

        centroid = np.nanmean(cluster_X, axis=0)
        median = np.nanmedian(cluster_X, axis=0)
        ax.plot(x, centroid, color="black", linewidth=centroid_lw, label="Mean (Centroid)")
        ax.plot(x, median, color="black", linewidth=centroid_lw, linestyle="--", label="Median")

        ax.set_title(f"Cluster {cluster} (n={mask.sum()})", fontsize=12, fontweight="bold")
        ax.grid(True, alpha=0.2)
        ax.legend(loc="upper right")

    for ax in axes[n_clusters:]:
        ax.axis("off")

    fig.suptitle(f"{clustering.split('/')[-1]}", fontsize=15, fontweight="bold")
    fig.supxlabel("Date")
    fig.supylabel("Value")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Example:
# plot_clusters_with_centroids(CLUSTERINGS[0])

In [ ]:
data_df = pd.read_csv(DATA)
data_df.columns = data_df.columns.str.strip()

if ID_COL not in data_df.columns:
    raise ValueError(f"ID column '{ID_COL}' not found in data file: {DATA}")

date_cols = [
    c for c in data_df.columns
    if c != ID_COL and pd.notna(pd.to_datetime(c, format="%Y-%m-%d", errors="coerce"))
]
if len(date_cols) == 0:
    raise ValueError("No date columns were found in the data file.")

X_df = data_df[[ID_COL] + date_cols].copy()
X_df[ID_COL] = X_df[ID_COL].astype(str)


results = []
label_vectors = {}

for clustering in CLUSTERINGS:
    print(f"Processing clustering: {clustering}")

    if not Path(clustering).exists():
        print(f"  Skipping: file not found -> {clustering}")
        continue

    try:
        labels_df = load_cluster_labels(clustering)
    except Exception as exc:
        print(f"  Skipping: failed to load labels ({exc})")
        continue

    merged = X_df[[ID_COL]].merge(labels_df, on=ID_COL, how="inner")
    if len(merged) == 0:
        print("  Skipping: no overlapping IDs between data and clustering labels")
        continue

    aligned_X = X_df.set_index(ID_COL).loc[merged[ID_COL], date_cols].to_numpy(dtype=float)
    cluster_labels = merged["cluster_label"].to_numpy()

    n_unique = len(pd.unique(cluster_labels))
    if n_unique < 2 or n_unique >= len(cluster_labels):
        silhouette_avg = np.nan
        print("  Silhouette Coefficient: NaN (requires 2..n-1 clusters)")
    else:
        silhouette_avg = float(silhouette_score(aligned_X, cluster_labels))
        print(f"  Silhouette Coefficient: {silhouette_avg:.6f}")

    results.append(
        {
            "clustering": clustering,
            "n_samples": int(len(merged)),
            "n_clusters": int(n_unique),
            "silhouette": silhouette_avg,
        }
    )
    label_vectors[clustering] = merged[[ID_COL, "cluster_label"]].copy()
    plot_clusters_with_centroids(clustering)

if len(results) == 0:
    print("No valid clustering results were computed.")
else:
    print("\nSummary")
    print(pd.DataFrame(results).to_string(index=False))

Processing clustering: ../data/outputs/agp_forecast_case5_clusters_cluster_specific_combined.csv
  Silhouette Coefficient: 0.008666
